In [ ]:
# Instalaivimas reikalingų paketų
!pip install git+https://github.com/huggingface/transformers accelerate
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
#Qwen/Qwen3-VL-2B-Instruct
#Qwen/Qwen3-VL-32B-Instruct

model = Qwen3VLForConditionalGeneration.from_pretrained(
     "Qwen/Qwen3-VL-32B-Instruct",
     dtype=torch.bfloat16,
     attn_implementation="flash_attention_2",
     device_map="auto",
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-32B-Instruct")
print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_edited.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        messages = [
            {
                "role": "user",
                "content": [
            {
                "type": "image",
                "image": image,
            },
                {"type": "text", "text": "Parašyk trumpai. Ne daugiau 10 žodžių. Tekstas yra paveikslėlio antraštė. Nerašyk jokio papildomo teksto."},
            ],
            }
        ]

    # Preparation for inference
        inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
        )
        inputs = inputs.to(model.device)

    # Inference: Generation of the output
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        generated_caption = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        generated_caption = str(generated_caption)
        generated_caption = re.sub(r'[^\w\s]', '', generated_caption)

        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_lt_18.csv", sep=';', index=False)
print("Generavimas baigtas.")